Comenzamos cargando las importaciones que seran necesarias en la realización de este proceso de procesado de los datos referentes al archivo de la "NASA exoplanet archive" que hemos ya preseleccionado para tener un tamaño de datios acorde para este proyecto sin ser este excesibamente masibo.

In [ ]:
import pandas as pd
from pathlib import Path

Se crean las bariables que nos daran aceso a cargar y guardar nuestros archivos .cvs

In [ ]:
# Configuraciones de rutas (pathlib resuelve conflictos rutas linux y windows)
BASE_PATH = Path().resolve()
DATA_RAW = BASE_PATH / ".." / "data" / "raw"
DATA_PROCESSED = BASE_PATH / ".." / "data" / "processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

file_path = DATA_RAW / "nasa_exoplanets.csv"

In [61]:
# Lectura del arcchivo csv raw
df = pd.read_csv(file_path)

# Informacion del dataframe
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

# Impresion de primeras filas
df.head()

Filas: 3463
Columnas: 8


,pl_name,hostname,ra,dec,sy_dist,pl_rade,pl_bmasse,pl_orbsmax
0,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.78,NaN,0.0116
1,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.87,NaN,0.0113
2,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,NaN,0.705944,NaN
3,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,1.24,NaN,NaN
4,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.76,NaN,NaN


### Limpieza de posibles espacios en blanco

Con tal de asegurar una correcta limpieza posterior de posibles valores duplicados se realizada una eliminacion de posibles espacios vacios que pudieran en un futuro distorsionar una busqueda por nombre de planeta o por nombre de sistema estelar.

In [62]:
df["pl_name"] = (
    df["pl_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df["hostname"] = (
    df["hostname"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

### Encontrada limitación
Los valores pueden variar entre observaciones debido a diferentes metodologías, por lo que la media representa una aproximación simplificada para los multiples valores o nulos y para eliminar filas duplicados (tecnicamente no estan duplicadas pero un mismo planeta tiene diferentes observaciones por lo que ocupa multiples filas del dataset).

In [63]:
df[df["pl_name"] == "Kepler-42 b"]

,pl_name,hostname,ra,dec,sy_dist,pl_rade,pl_bmasse,pl_orbsmax
0,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.780,NaN,0.01160
1,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.870,NaN,0.01130
2,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,NaN,0.705944,NaN
3,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,1.240,NaN,NaN
4,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.760,NaN,NaN
5,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.912,NaN,NaN
6,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,3.460,NaN,0.01900
7,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.870,NaN,0.01130
8,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.780,NaN,0.01130
9,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,0.870,NaN,0.01130


### Solución
Dado que un mismo exoplaneta puede presentar múltiples registros derivados de diferentes observaciones y técnicas de medición, se ha optado por agrupar los datos por identificador de planeta (pl_name) y calcular valores medios para las variables numéricas.

Este enfoque permite reducir la redundancia del dataset y obtener una representación única por planeta, facilitando su análisis posterior, aunque implica una simplificación de la variabilidad observacional, pero esta es necesaria.

In [64]:
df = df.groupby("pl_name").agg({
    "hostname": "first",
    "ra": "mean",
    "dec": "mean",
    "sy_dist": "mean",
    "pl_rade": "mean",
    "pl_bmasse": "mean",
    "pl_orbsmax": "mean"
}).reset_index()

df[df["pl_name"] == "Kepler-42 b"]

,pl_name,hostname,ra,dec,sy_dist,pl_rade,pl_bmasse,pl_orbsmax
880,Kepler-42 b,Kepler-42,292.2196,44.617367,40.0595,1.129455,0.705944,0.012476


In [65]:
print("Filas:", len(df))
print("Planetas:", df["pl_name"].nunique())

Filas: 1183
Planetas: 1183


In [52]:
# Informacion basica del dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1183 entries, 0 to 1182
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pl_name     1183 non-null   str    
 1   hostname    1183 non-null   str    
 2   ra          1183 non-null   float64
 3   dec         1183 non-null   float64
 4   sy_dist     1183 non-null   float64
 5   pl_rade     365 non-null    float64
 6   pl_bmasse   1085 non-null   float64
 7   pl_orbsmax  1133 non-null   float64
dtypes: float64(6), str(2)
memory usage: 74.1 KB


Aqui se exploraran datos muy valiosos sobre el dataframe, desde datos minimos de las diferentes variables a datos medios por porcentajes.

In [36]:
# Descropcion tecnica del dataframe
df.describe()

,ra,dec,sy_dist,pl_rade,pl_bmasse,pl_orbsmax
count,3463.000000,3463.000000,3463.000000,1182.000000,2607.000000,2630.000000
mean,182.474188,-3.212466,33.595778,3.029052,795.890206,26.573603
std,105.093231,39.971338,17.376655,5.436360,1554.138400,506.664374
min,0.185606,-84.231749,1.301190,0.270000,0.100000,0.005626
25%,88.517219,-34.997765,18.270200,1.360000,8.700000,0.066520
50%,175.550536,-2.144437,34.193700,1.997830,162.093300,0.246500
75%,284.059250,28.329815,47.289900,2.728154,823.135000,2.114750
max,355.781346,84.333761,64.994300,90.810000,24790.615746,19000.000000


Comenzamos contamos los valores nulos y los valores duplicados.

Terminamos eliminando los valores duplicados de nuestro dataframe, seleccionados minuciosamente por nombre de planeta, ya que la funcion de eliminacion de duplicados discrimina y elimina muchos registros por ser muy parecidos.

In [53]:
print("\nValores nulos por columna:\n")
print(df.isnull().sum())

print("\nTotal de valores nulos:", df.isnull().sum().sum())

print("\nFilas duplicadas:", df.duplicated().sum())

print("Filas antes:", len(df))

df = df.drop_duplicates(subset=["pl_name"])

print("Filas después:", len(df))


Valores nulos por columna:

pl_name         0
hostname        0
ra              0
dec             0
sy_dist         0
pl_rade       818
pl_bmasse      98
pl_orbsmax     50
dtype: int64

Total de valores nulos: 966

Filas duplicadas: 0
Filas antes: 1183
Filas después: 1183


# Revisión de valores

Se revisan los valores referentes de la posicion espacial ra, dec y la distancia con la tierra para explorar la necesidad de eliminar valores fuera de rango, outliners o posibles errores sin sentido como una dec de mas de 90 grados saluendose del plano visual.

In [54]:
print("Inicial:", len(df))

# RA
df = df[(df["ra"] >= 0) & (df["ra"] <= 360)]
print("Tras filtro RA:", len(df))

# DEC
df = df[(df["dec"] >= -90) & (df["dec"] <= 90)]
print("Tras filtro DEC:", len(df))

# Distancia
df = df[df["sy_dist"] > 0]
print("Tras filtro distancia:", len(df))

Inicial: 1183
Tras filtro RA: 1183
Tras filtro DEC: 1183
Tras filtro distancia: 1183


Antes de proceder a guardar todo el dataframe como un archivo cvs procesado, se revisa cuantos sistemas diferentes hay y cuantos planetas tenemos

In [56]:
print("Numero de registros:", len(df))
print("Número de sistemas:", df["hostname"].nunique())
print("Número de planetas:", df["pl_name"].nunique())

Numero de registros: 1183
Número de sistemas: 773
Número de planetas: 1183


In [57]:
output_path = DATA_PROCESSED / "nasa_exoplanets_processed.csv"

df.to_csv(output_path, index=False)

print("Guardado en:", output_path)

Guardado en: F:\Usuario\Diego\Estudios\Ilerna\EspacializacionBigDataIA\ProyectoBigData\proyectoBCSS\notebook\..\data\processed\nasa_exoplanets_processed.csv
